<a href="https://colab.research.google.com/github/emna378/ids-explainability-audit/blob/emna/01_donnees_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q shap

Mounted at /content/drive


In [4]:
import os, random, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "/content/drive/MyDrive/dataset"
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

# 1 — Charger les données

In [5]:
tr = pd.read_csv(f"{DATA_DIR}/UNSW_NB15_training-set.csv")
te = pd.read_csv(f"{DATA_DIR}/UNSW_NB15_testing-set.csv")
df = pd.concat([tr, te], ignore_index=True)     # on colle les deux fichiers l'un sous l'autre
print("Taille (lignes, colonnes) :", df.shape)
print(df.head())

Taille (lignes, colonnes) : (257673, 45)
   id       dur proto service state  spkts  dpkts  sbytes  dbytes  \
0   1  0.000011   udp       -   INT      2      0     496       0   
1   2  0.000008   udp       -   INT      2      0    1762       0   
2   3  0.000005   udp       -   INT      2      0    1068       0   
3   4  0.000006   udp       -   INT      2      0     900       0   
4   5  0.000010   udp       -   INT      2      0    2126       0   

          rate  ...  ct_dst_sport_ltm  ct_dst_src_ltm  is_ftp_login  \
0   90909.0902  ...                 1               2             0   
1  125000.0003  ...                 1               2             0   
2  200000.0051  ...                 1               3             0   
3  166666.6608  ...                 1               3             0   
4  100000.0025  ...                 1               3             0   

   ct_ftp_cmd  ct_flw_http_mthd  ct_src_ltm  ct_srv_dst  is_sm_ips_ports  \
0           0                 0          

# 2. Explorer les données









In [6]:
print(df.dtypes.value_counts())
print("Normal (0) vs attaque (1) :")
print(df["label"].value_counts())
print("Part d'attaques :", round(df["label"].mean(), 3))
print()


if "attack_cat" in df.columns:
    print(df["attack_cat"].value_counts())
elif 'family' in globals():
    print("Familles d'attaques (depuis la variable 'family') :")
    print(family.value_counts())
else:
    print("La colonne 'attack_cat' n'est pas disponible dans le DataFrame.")

print()
print(df[["proto", "service", "state"]].nunique())   # nombre de valeurs différentes

int64      30
float64    11
object      4
Name: count, dtype: int64
Normal (0) vs attaque (1) :
label
1    164673
0     93000
Name: count, dtype: int64
Part d'attaques : 0.639

attack_cat
Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174
Name: count, dtype: int64

proto      133
service     13
state       11
dtype: int64


# 3 — Nettoyer les données

In [7]:
family = df["attack_cat"].copy()
df = df.drop(columns=["id", "attack_cat"])
df = df.replace([np.inf, -np.inf], np.nan)

n_manquantes = int(df.isna().sum().sum())
n_doublons = int(df.duplicated().sum())
print("Valeurs manquantes :", n_manquantes)
print("Lignes en double   :", n_doublons, "sur", len(df))

Valeurs manquantes : 0
Lignes en double   : 103575 sur 257673


## on enlève les doublons

In [8]:
df = df.drop_duplicates()
family = family.loc[df.index]
print("Taille après nettoyage :", df.shape)

Taille après nettoyage : (154098, 43)


# 4 — Découper en entraînement, validation et test

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["label"])
y = df["label"]

# 1) on met 10 % de côté pour le TEST (on n'y touchera qu'à la fin)
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

# 2) on coupe le reste : 90 % entraînement, 10 % validation (= 60 % et 20 % du total)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.10, stratify=y_tmp, random_state=SEED)
for nom, part in [("entraînement", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{nom:13s}: {len(part):7d} flux | part d'attaques = {part.mean():.3f}")

entraînement :  110950 flux | part d'attaques = 0.444
validation   :   12328 flux | part d'attaques = 0.444
test         :   30820 flux | part d'attaques = 0.444


# 5 — Préparer les variables

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

cat_cols = ["proto", "service", "state"]
num_cols = [c for c in X_train.columns if c not in cat_cols]
print(len(num_cols), "colonnes numériques,", len(cat_cols), "colonnes de texte")

# Pour la régression logistique : numériques mises à la même échelle + texte en 0/1
prep_lin = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])

# Pour les arbres : numériques inchangées + texte remplacé par un numéro
prep_tree = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)])

39 colonnes numériques, 3 colonnes de texte


# 6 — Mesurer et entraîner la baseline

In [11]:
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score, recall_score)
def evaluate(y_true, proba, thr=0.5, name="modèle", verbose=True):
    """Calcule les métriques pour un seuil donné. proba = probabilité d'attaque (0 à 1)."""
    pred = (proba >= thr).astype(int)
    res = {"modèle": name, "seuil": round(float(thr), 3),
           "PR-AUC": round(average_precision_score(y_true, proba), 4),
           "précision": round(precision_score(y_true, pred), 4),
           "rappel": round(recall_score(y_true, pred), 4),
           "F1": round(f1_score(y_true, pred), 4)}
    if verbose:
        print(f"=== {name} (seuil = {thr:.2f}) ===")
        print("Matrice [[vrais normaux, fausses alertes], [attaques ratées, attaques détectées]]")
        print(confusion_matrix(y_true, pred))
        print(classification_report(y_true, pred, digits=3, target_names=["normal", "attaque"]))
        print("PR-AUC :", res["PR-AUC"])
    return res
def best_threshold(y_true, proba):
    """Seuil qui donne le meilleur F1. À appeler UNIQUEMENT sur la validation."""
    pr, rc, th = precision_recall_curve(y_true, proba)
    f1 = 2 * pr[:-1] * rc[:-1] / (pr[:-1] + rc[:-1] + 1e-12)
    return float(th[np.argmax(f1)])

 La baseline : régression logistique

In [12]:
from sklearn.linear_model import LogisticRegression
baseline = Pipeline([
    ("prep", prep_lin),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
t0 = time.time()
baseline.fit(X_train, y_train)
print("Temps d'entraînement :", round(time.time() - t0, 1), "s")
p_val_base = baseline.predict_proba(X_val)[:, 1]
evaluate(y_val, p_val_base, thr=0.5, name="Baseline (seuil 0.5)")

Temps d'entraînement : 14.0 s
=== Baseline (seuil 0.5) (seuil = 0.50) ===
Matrice [[vrais normaux, fausses alertes], [attaques ratées, attaques détectées]]
[[5203 1655]
 [ 261 5209]]
              precision    recall  f1-score   support

      normal      0.952     0.759     0.845      6858
     attaque      0.759     0.952     0.845      5470

    accuracy                          0.845     12328
   macro avg      0.856     0.855     0.845     12328
weighted avg      0.866     0.845     0.845     12328

PR-AUC : 0.9275


{'modèle': 'Baseline (seuil 0.5)',
 'seuil': 0.5,
 'PR-AUC': np.float64(0.9275),
 'précision': 0.7589,
 'rappel': 0.9523,
 'F1': 0.8447}